# ESS11 — Analyse de la confiance

**Source des données** : [European Social Survey (ESS), Round 11](https://ess.sikt.no/en/series/50ebb530-c72e-4002-a7bf-5581e3fdfe21) — sous-ensemble `ESS_start.csv` préparé à l'étape 1.

**Étape 2 — Structure factorielle de la confiance**

À partir du sous-ensemble `ESS_start.csv` (étape 1), on analyse les 10 items de confiance :
- **interpersonnelle** : `ppltrst`, `pplfair`, `pplhlp` ;
- **institutionnelle** : `trstprl`, `trstlgl`, `trstplc`, `trstplt`, `trstprt`, `trstep`, `trstun`.

Déroulé :
1. matrice de corrélations ;
2. factorisabilité (KMO, test de Bartlett) et choix du nombre de facteurs ;
3. analyse factorielle exploratoire (EFA) avec rotation oblique (*oblimin*) ;
4. sauvegarde des scores factoriels dans `ESS_start.csv` ;
5. scattergram des deux dimensions principales (moyennes par pays).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Compatibilité factor_analyzer / scikit-learn >= 1.6 :
# l'argument force_all_finite a été renommé ensure_all_finite
import sklearn.utils as _sku
_orig_check_array = _sku.check_array
def _check_array(*args, **kwargs):
    if "force_all_finite" in kwargs:
        kwargs["ensure_all_finite"] = kwargs.pop("force_all_finite")
    return _orig_check_array(*args, **kwargs)
_sku.check_array = _check_array

from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity
import factor_analyzer.factor_analyzer as _fam
_fam.check_array = _check_array

df = pd.read_csv("ESS_start.csv")

trust_vars = [
    "ppltrst", "pplfair", "pplhlp",                       # interpersonnelle
    "trstprl", "trstlgl", "trstplc",                      # institutionnelle
    "trstplt", "trstprt", "trstep", "trstun",
]

X = df[trust_vars].dropna()
print(f"Échantillon complet : {len(df):,} — après suppression des NA sur la confiance : {len(X):,}")

Échantillon complet : 46,162 — après suppression des NA sur la confiance : 40,602


## 1. Matrice de corrélations

In [ ]:
corr = X.corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title("Matrice de corrélations — items de confiance (ESS11)")
plt.tight_layout()
plt.show()

## 2. Factorisabilité et nombre de facteurs

In [ ]:
chi2, p_bartlett = calculate_bartlett_sphericity(X)
kmo_per_item, kmo_global = calculate_kmo(X)
print(f"Bartlett : chi2 = {chi2:,.0f}, p = {p_bartlett:.3g}")
print(f"KMO global : {kmo_global:.3f}")
print(pd.Series(kmo_per_item, index=trust_vars).round(3).rename("KMO par item"))

# Valeurs propres (critère de Kaiser + scree plot)
fa_expl = FactorAnalyzer(n_factors=len(trust_vars), rotation=None)
fa_expl.fit(X)
eigenvalues, _ = fa_expl.get_eigenvalues()
print("\nValeurs propres :", np.round(eigenvalues, 2))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(eigenvalues) + 1), eigenvalues, "o-")
ax.axhline(1, color="red", ls="--", lw=1, label="Kaiser (λ = 1)")
ax.set_xlabel("Facteur")
ax.set_ylabel("Valeur propre")
ax.set_title("Scree plot")
ax.legend()
plt.tight_layout()
plt.show()

## 3. EFA avec rotation oblique (oblimin)

On retient **2 facteurs** (critère de Kaiser et théorie : confiance interpersonnelle vs institutionnelle). La rotation oblique autorise la corrélation entre facteurs.

In [ ]:
fa = FactorAnalyzer(n_factors=2, rotation="oblimin", method="minres")
fa.fit(X)

loadings = pd.DataFrame(fa.loadings_, index=trust_vars,
                        columns=["F1", "F2"]).round(3)
communalities = pd.Series(fa.get_communalities(), index=trust_vars,
                          name="h²").round(3)
print("Saturations (pattern matrix) :")
print(loadings.join(communalities))

var_exp = pd.DataFrame(fa.get_factor_variance(),
                       index=["SS loadings", "Prop. variance", "Cumul"],
                       columns=["F1", "F2"]).round(3)
print("\nVariance expliquée :")
print(var_exp)

phi = pd.DataFrame(fa.phi_, index=["F1", "F2"], columns=["F1", "F2"]).round(3)
print("\nCorrélation entre facteurs (phi) :")
print(phi)

In [ ]:
# Visualisation des saturations
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(loadings, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title("Saturations factorielles (rotation oblimin)")
plt.tight_layout()
plt.show()

## 4. Scores factoriels et sauvegarde dans `ESS_start.csv`

Les scores (méthode de régression) sont calculés pour les répondants sans valeur manquante sur les 10 items, puis ajoutés au fichier. On nomme les facteurs d'après les saturations : `trust_inst` (confiance institutionnelle) et `trust_interp` (confiance interpersonnelle).

In [ ]:
scores = fa.transform(X)

# Identifier quel facteur est 'institutionnel' : celui qui sature le plus trstprl
f_inst = loadings.loc["trstprl"].abs().idxmax()
f_interp = "F2" if f_inst == "F1" else "F1"
mapping = {f_inst: "trust_inst", f_interp: "trust_interp"}
print(f"{f_inst} -> trust_inst | {f_interp} -> trust_interp")

scores_df = pd.DataFrame(scores, index=X.index, columns=["F1", "F2"]).rename(columns=mapping)

df["trust_inst"] = scores_df["trust_inst"]
df["trust_interp"] = scores_df["trust_interp"]

df.to_csv("ESS_start.csv", index=False)
print(f"ESS_start.csv mis à jour : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
df[["cntry", "trust_inst", "trust_interp"]].describe().round(3)

## 5. Scattergram des deux dimensions — moyennes par pays

Moyennes pondérées par `anweight` (pondération d'analyse recommandée par l'ESS).

In [ ]:
def wmean(g, col):
    m = g[col].notna() & g["anweight"].notna()
    return np.average(g.loc[m, col], weights=g.loc[m, "anweight"])

pays = (df.groupby("cntry")
          .apply(lambda g: pd.Series({
              "trust_interp": wmean(g, "trust_interp"),
              "trust_inst": wmean(g, "trust_inst"),
          }), include_groups=False)
          .reset_index())

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(pays["trust_interp"], pays["trust_inst"], s=60, color="steelblue", zorder=3)
for _, row in pays.iterrows():
    ax.annotate(row["cntry"], (row["trust_interp"], row["trust_inst"]),
                textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.axhline(0, color="grey", lw=0.8, ls="--")
ax.axvline(0, color="grey", lw=0.8, ls="--")
ax.set_xlabel("Confiance interpersonnelle (score factoriel moyen)")
ax.set_ylabel("Confiance institutionnelle (score factoriel moyen)")
ax.set_title("Confiance interpersonnelle vs institutionnelle — moyennes par pays (ESS11, pondéré anweight)")
plt.tight_layout()
plt.show()

print(f"Corrélation agrégée entre les deux dimensions (niveau pays) : "
      f"{pays['trust_interp'].corr(pays['trust_inst']):.3f}")